<a href="https://colab.research.google.com/github/raimund-buehler/visualizing-the-brain/blob/main/notebooks/day_one_de.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Tag Eins in Colab öffnen"/></a>

# Tag Eins: Von Gehirnbildern zu einem ersten fMRT-Vergleich

Heute verwenden wir Python, um Gehirnbilder zu öffnen, die Zahlen darin anzuschauen und eine einfache Frage an echte fMRT-Daten zu stellen.

Am Ende kannst du:

- Voxel, Schichten, Volumen und 4D-fMRT-Daten beschreiben,
- ein MRT-Bild als Zahlenfeld untersuchen,
- dich mit verschiedenen Darstellungen durch das Gehirn bewegen,
- erklären, warum ein fMRT-Bild immer mit etwas verglichen werden muss,
- die Grundidee eines GLM und eines Kontrasts beschreiben, und
- einen First-Level-Kontrast für eine Person berechnen.

## Was ist ein Jupyter Notebook?

Ein Jupyter Notebook ist eine Seite, auf der Text, Code, Bilder und Ergebnisse zusammenleben können.

Dieses Notebook hat zwei Arten von Kästchen. Sie heißen **Zellen**:

- **Textzellen**, wie diese hier, erklären, was passiert.
- **Codezellen** enthalten Python-Anweisungen, die der Computer ausführen kann.

Um eine Codezelle auszuführen, klicke links auf den Play-Knopf. Du kannst auch in die Zelle klicken und **Shift + Enter** drücken.

Führe die Zellen von oben nach unten aus. Eine spätere Zelle braucht manchmal eine Variable, die in einer früheren Zelle erstellt wurde.

## Setup: hier zuerst auf Play drücken

Die nächste Zelle installiert und importiert unsere Werkzeuge. Sie lädt außerdem eine anatomische Standard-Gehirnvorlage.

Du musst noch nicht jede Zeile im Setup verstehen. Führe sie einmal aus und warte auf **Setup abgeschlossen**.

In [ ]:
# @title Setup: Werkzeuge installieren und anatomische Vorlage laden
import sys
import subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "nilearn", "nibabel", "matplotlib", "numpy", "pandas"
    ])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nilearn import datasets, image, plotting
from nilearn.glm.first_level import (
    FirstLevelModel,
    make_first_level_design_matrix,
    spm_hrf,
)

%matplotlib inline

# Das ist ein durchschnittliches anatomisches Gehirn in einem gemeinsamen Koordinatensystem.
anat_img = datasets.load_mni152_template(resolution=2)

print("Setup abgeschlossen.")
print("Form des anatomischen Bildes:", anat_img.shape)

## MRT-Daten: Anatomie und Funktion

MRT-Scanner können verschiedene Arten von Daten erzeugen. Heute verwenden wir zwei davon:

- Ein **anatomisches MRT** ist ein detailliertes 3D-Bild der Gehirnstruktur. Jedes Voxel hat einen Helligkeitswert.
- **fMRT** nimmt viele 3D-Gehirnvolumen nacheinander auf. Jedes Voxel hat deshalb eine Reihe von Werten über die Zeit.

Ein Scanner erfasst das Gehirn in **Schichten**. Ein Stapel von Schichten ergibt ein 3D-**Volumen**. Bei fMRT ergeben viele Volumen zusammen einen 4D-Datensatz.

<img src="https://raw.githubusercontent.com/raimund-buehler/visualizing-the-brain/main/src/images/mri_fmri_build_up.png" alt="MRT- und fMRT-Daten entstehen aus Voxeln, Schichten, Volumen und Zeit" width="850">

Der grün markierte Würfel ist ein **Voxel**. Ein Voxel ist wie ein 3D-Pixel.

## Ein Gehirnbild ist ein Feld aus Zahlen

Ein Computer beginnt nicht mit einem Bild von einem Gehirn. Er speichert ein Raster aus Zahlen. So ein Zahlenraster nennt man ein **Array**. Plot-Programme verwandeln diese Zahlen in Helligkeit oder Farbe.

Bei einem anatomischen 3D-Bild hat das Array drei Dimensionen:

- die erste Position bewegt sich durch die **x**-Richtung,
- die zweite durch die **y**-Richtung, und
- die dritte durch die **z**-Richtung.

Die `shape` eines Bildes sagt uns, wie viele Voxel-Positionen es entlang jeder Dimension gibt.

### Das Gehirnbild als Array öffnen

Die Setup-Zelle hat ein anatomisches Gehirnbild geladen und in der Variable `anat_img` gespeichert.

Mit `get_fdata()` holen wir die Bilddaten als Zahlen aus dem Bild. Danach können wir Python fragen, welche Form dieses Zahlenfeld hat.

In [ ]:
# Speichere die Zahlen aus dem Bild in der Variable anat_data.
anat_data = anat_img.get_fdata()

print("Python-Objekt:", type(anat_img))
print("Array-Typ:", type(anat_data))
print("Array-Form:", anat_data.shape)
print("Anzahl der Dimensionen:", anat_data.ndim)

### Ein einzelnes Voxel lesen

Wir können drei Array-Positionen verwenden, um nach dem Wert eines einzelnen Voxels zu fragen.

Der Code unten findet automatisch die mittlere Array-Position. Die drei Zahlen sind ein **Array-Index**. Sie bezeichnen ein gespeichertes Voxel.

In [ ]:
# Finde den mittleren Index entlang x, y und z.
middle_x = anat_data.shape[0] // 2
middle_y = anat_data.shape[1] // 2
middle_z = anat_data.shape[2] // 2

middle_value = anat_data[middle_x, middle_y, middle_z]

print("Mittlerer Array-Index:", (middle_x, middle_y, middle_z))
print("Dort gespeicherter Wert:", middle_value)

Ein Array kann uns auch einen kleinen Block von benachbarten Voxeln geben. Der Doppelpunkt in `a:b` bedeutet: starte bei `a` und stoppe kurz vor `b`.

Zuerst wählen wir einen `3 × 3 × 3`-Block um das mittlere Voxel herum aus. Danach zeichnen wir ihn als kleines 3D-Objekt.

In [ ]:
# Wähle einen 3 × 3 × 3-Block um die Mitte herum aus.
small_block = anat_data[
    middle_x - 1:middle_x + 2,
    middle_y - 1:middle_y + 2,
    middle_z - 1:middle_z + 2,
]

print("Block-Form:", small_block.shape)
print(small_block)

### Aus Zahlen wird ein 3D-Objekt

Die nächste Zelle zeichnet den ausgewählten `3 × 3 × 3`-Block als 27 Würfel.

Jeder Würfel ist ein Voxel. Seine Farbe kommt von der Zahl, die an dieser Position gespeichert ist: dunkles Violett bedeutet einen niedrigeren Wert, helles Gelb einen höheren Wert.

In [ ]:
# Skaliere die Werte zwischen 0 und 1, damit sie zu Farben werden können.
block_range = np.ptp(small_block)
scaled_values = (small_block - small_block.min()) / block_range
voxel_colors = plt.cm.viridis(scaled_values)

# Zeichne alle ausgewählten Array-Positionen als gefüllte Würfel.
fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111, projection="3d")
ax.voxels(
    np.ones(small_block.shape, dtype=bool),
    facecolors=voxel_colors,
    edgecolor="black",
)
ax.set_xlabel("x-Position")
ax.set_ylabel("y-Position")
ax.set_zlabel("z-Position")
ax.set_title(f"Ein {small_block.shape}-Array als Voxel gezeichnet")
ax.set_box_aspect((1, 1, 1))
plt.show()

### Gemeinsam nachdenken

Besprecht diese Fragen mit einer Nachbarin oder einem Nachbarn, bevor ihr weitermacht:

1. Wie würdest du den Code ändern, damit ein `4 × 4 × 4`-Block entsteht?
2. Was bedeuten die Farben im Plot?

<details>
<summary>Mögliche Antworten</summary>

1. Eine Möglichkeit ist, den Stoppwert auf `middle_x + 3`, `middle_y + 3` und `middle_z + 3` zu ändern. Dann sind vier Positionen enthalten: `-1`, `0`, `+1` und `+2` um die Mitte herum.
2. Die Farben stehen für die Zahlen, die in den Voxeln gespeichert sind. In diesem anatomischen Bild sind diese Zahlen Bildintensitäten: Sie sagen dem Plot-Programm, wie hell oder dunkel jedes Voxel sein soll. Sie sind **noch keine Aktivierung**. Später, wenn wir fMRT-Kontrastkarten plotten, bedeuten Farben etwas anderes: mehr Signal für eine Aufgabenbedingung im Vergleich zu einer anderen Aufgabenbedingung.

</details>

## Gehirnschichten und Richtungen

Wissenschaftlerinnen und Wissenschaftler betrachten das 3D-Array als Schichten aus drei Richtungen:

- **Sagittale** Schichten trennen links und rechts. Nilearn nennt das die `x`-Richtung.
- **Koronale** Schichten trennen vorne und hinten. Nilearn nennt das die `y`-Richtung.
- **Axiale** Schichten trennen oben und unten. Nilearn nennt das die `z`-Richtung.

<img src="https://raw.githubusercontent.com/raimund-buehler/visualizing-the-brain/main/src/images/AnatomicalPlanes.png" alt="Sagittale, koronale und axiale Gehirnebenen" width="700">

## Das anatomische Bild plotten

Die Standardansicht `ortho` zeigt eine sagittale, eine koronale und eine axiale Schicht zusammen. Das Fadenkreuz markiert in allen drei Ansichten dieselbe Stelle.

In [ ]:
plotting.plot_anat(
    anat_img,
    display_mode="ortho",
    title="Anatomisches MRT: drei Schichtrichtungen",
)
plotting.show()

### Axiale Schichten auswählen

Manchmal wollen wir mehrere Schichten aus derselben Richtung vergleichen. Hier fragt `display_mode="z"` nach **axialen** Schichten, also von unten im Gehirn nach oben.

Die Werte in `slice_heights` sind Höhen in Millimetern. Die Liste unten enthält tiefe, mittlere und hohe Schichten, damit du sehen kannst, wie sich das Gehirn von unten nach oben verändert.

Diese Kartenkoordinaten sind etwas anderes als die Array-Indizes von vorhin. Nilearn liest die Bildinformationen aus, um die Voxel im Gehirnraum zu platzieren.

In [ ]:
# Höhen der z-Schichten, von unten nach oben geordnet.
slice_heights = [-30, -10, 10, 30, 50, 70]

plotting.plot_anat(
    anat_img,
    display_mode="z",
    cut_coords=slice_heights,
    title="Axiale Schichten von tieferen zu höheren Positionen",
)
plotting.show()

## Das Gehirn interaktiv erkunden

`view_img` erstellt eine Ansicht, in die du klicken kannst. Klicke in die verschiedenen Ansichten und beobachte, wie sich das Fadenkreuz und die `x`-, `y`- und `z`-Koordinaten verändern.

In [ ]:
anat_viewer = plotting.view_img(
    anat_img,
    bg_img=False,
    cmap="gray",
    colorbar=False,
    symmetric_cmap=False,
    title="Klicke, um die anatomische Vorlage zu erkunden",
)

anat_viewer

### Gehirn-Schatzsuche

Benutze die interaktive Ansicht, um dich in die Nähe des folgenden ungefähren Orientierungspunktes zu bewegen. Die Koordinaten sind Hinweise, keine exakten Grenzen: Gehirnareale sind größer als ein einzelner Punkt und unterscheiden sich zwischen Menschen.

**Oberer/parietaler Kortex:** Bewege dich in Richtung `(x=-40, y=-25, z=60)`. Das liegt hoch an der Seite des Gehirns, in der Nähe von Regionen, die mit Körperempfindung und Handbewegung zu tun haben.

Beschreibe die Lage mit normalen Wörtern wie **links/rechts**, **vorne/hinten** und **oben/unten**. Es geht nicht darum, ein perfektes Voxel zu finden.

## Einen Beispiel-fMRT-Scan laden

Um echte 4D-Daten zu erkunden, laden wir einen kleinen Beispielscan, den Nilearn bereitstellt. Die nächste Zelle lädt die Dateien beim ersten Ausführen herunter. Zuerst untersuchen wir die Datenform und die Zeitreihe eines Voxels. Danach schauen wir uns an, was die Versuchsperson während des Scans gemacht hat.

In [ ]:
# Lade das fMRT-Bild und die Ereignistabelle einer Versuchsperson herunter.
fmri_dataset = datasets.fetch_localizer_first_level()
fmri_img = image.load_img(fmri_dataset.epi_img)
events = pd.read_table(fmri_dataset.events)

print("fMRT-Bildform:", fmri_img.shape)
print("Anzahl der Dimensionen:", len(fmri_img.shape))

## fMRT fügt eine vierte Dimension hinzu: Zeit

Ein anatomisches Bild speichert an jeder `(x, y, z)`-Voxelposition einen Wert. Ein fMRT-Bild fügt eine vierte Position hinzu: `t` für **Zeit**.

Wir können dieselben 4D-Daten auf zwei nützliche Arten betrachten:

1. als Abfolge von 3D-Gehirnvolumen, wie Bilder in einem Film, oder
2. als Zeitreihe aus jedem Voxel, die zeigt, wie sich sein Signal während des Scans verändert.

<img src="https://raw.githubusercontent.com/raimund-buehler/visualizing-the-brain/main/src/images/fmri_4d_timeseries.png" alt="Eine Abfolge von fMRT-Volumen und eine Zeitreihe aus einem Voxel" width="700">

Die **Repetitionszeit**, kurz **TR**, ist die Zeit zwischen einem ganzen Gehirnvolumen und dem nächsten. In unserem Beispiel nimmt der Scanner alle 2,4 Sekunden ein neues Volumen auf.

Die Form `(53, 63, 46, 128)` bedeutet:

- 53 Voxelpositionen entlang der ersten räumlichen Richtung,
- 63 entlang der zweiten,
- 46 entlang der dritten, und
- 128 Zeitpunkte.

Die ersten drei Zahlen beschreiben ein 3D-Volumen. Die vierte Zahl sagt uns, wie viele Volumen über die Zeit aufgenommen wurden.

### Die Zeitreihe eines Voxels extrahieren

Die Abbildung oben zeigt eine Linie, die aus einem Voxel kommt. Jetzt erzeugen wir so eine Linie aus echten Daten.

Um ein Voxel auszuwählen, wählen wir eine `x`-, `y`- und `z`-Arrayposition. Der letzte `:` bedeutet: **behalte alle Zeitpunkte**:

```python
fmri_data[x, y, z, :]
```

Wir starten mit einem Voxel nahe der Bildmitte.

In [ ]:
# Wähle die mittlere Array-Position im fMRT-Bild.
voxel_x = fmri_img.shape[0] // 2
voxel_y = fmri_img.shape[1] // 2
voxel_z = fmri_img.shape[2] // 2

# Lies dieses Voxel an jedem Zeitpunkt aus.
voxel_signal = np.asarray(
    fmri_img.dataobj[voxel_x, voxel_y, voxel_z, :]
)

print("Voxel-Index:", (voxel_x, voxel_y, voxel_z))
print("Form der Zeitreihe:", voxel_signal.shape)
print("Erste 10 Signalwerte:", voxel_signal[:10])

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(voxel_signal, color="royalblue")
plt.xlabel("Zeitpunkt (Volumennummer)")
plt.ylabel("fMRT-Signalintensität")
plt.title("Signal über die Zeit aus einem Voxel")
plt.grid(alpha=0.25)
plt.show()

### Probiere ein anderes Voxel aus

Ein Voxel liefert uns eine Zeitreihe. Aber das Gehirnbild enthält viele tausend Voxel. Wir bewegen uns ein kleines Stück vom mittleren Voxel weg und vergleichen das Signal.

In der nächsten Zelle kannst du die `+ 5`-Werte verändern. Halte die Änderungen klein, zum Beispiel `+ 3`, `+ 5` oder `- 5`, damit du in der Nähe des ursprünglichen Voxels bleibst.

In [ ]:
# Probiere ein anderes nahegelegenes Voxel aus.
# Verändere diese kleinen Verschiebungen und führe die Zelle erneut aus.
new_voxel_x = voxel_x + 5
new_voxel_y = voxel_y
new_voxel_z = voxel_z

new_voxel_signal = np.asarray(
    fmri_img.dataobj[new_voxel_x, new_voxel_y, new_voxel_z, :]
)

plt.figure(figsize=(10, 4))
plt.plot(voxel_signal, color="royalblue", label="mittleres Voxel")
plt.plot(new_voxel_signal, color="darkorange", label="nahegelegenes Voxel")
plt.xlabel("Zeitpunkt (Volumennummer)")
plt.ylabel("fMRT-Signalintensität")
plt.title("Zwei Zeitreihen aus nahegelegenen Voxeln")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

print("Mittleres Voxel:", (voxel_x, voxel_y, voxel_z))
print("Nahegelegenes Voxel:", (new_voxel_x, new_voxel_y, new_voxel_z))

### Diskutiert

1. Sehen die beiden Linien gleich, ähnlich oder sehr unterschiedlich aus?
2. Sind beide Linien glatt oder wirken sie verrauscht?
3. Warum könnten nahegelegene Voxel unterschiedliche Signale haben?
4. Warum sollten wir vorsichtig sein, wenn wir nur ein einzelnes Voxel interpretieren?

<details>
<summary>Mögliche Antworten</summary>

Nahegelegene Voxel können sich unterscheiden, weil sie unterschiedliches Gewebe, unterschiedliche Blutgefäße, unterschiedlich viel Rauschen oder verschiedene Mischungen von Gehirnsignalen enthalten. Ein einzelnes Voxel ist nützlich, um die Idee einer Zeitreihe zu verstehen. Eine echte fMRT-Analyse prüft aber viele Voxel und vergleicht Aufgabenbedingungen sorgfältig.

</details>

### Was ist das fMRT-Signal?

Zu jedem Zeitpunkt speichert der Scanner für jedes Voxel eine Signalintensität. Ein Teil dieser Zahl wird vom **BOLD-Signal** beeinflusst. BOLD steht für **blood-oxygen-level dependent**, also abhängig vom Sauerstoffgehalt im Blut.

Wenn eine Gruppe von Gehirnzellen aktiver wird, braucht sie Energie. Der Körper verändert dann lokal die Versorgung mit sauerstoffreichem Blut. fMRT ist empfindlich für diese Veränderungen im Blutsauerstoff. Es misst **nicht** direkt Gedanken oder elektrische Gehirnaktivität.

Die Linie eines Voxels kann sich auch durch Atmung, Herzschlag, Kopfbewegung, Scannerrauschen und langsame Veränderungen während des Scans verändern. Deshalb reicht ein einzelner Ausschlag in einer Zeitreihe nicht aus, um zu behaupten, dass eine Aufgabe dieses Voxel aktiviert hat.

## Ein einzelnes fMRT-Volumen anschauen

Die Zeitreihe hat ein Voxel über alle 128 Zeitpunkte verwendet. Wir können auch das Gegenteil machen: einen Zeitpunkt auswählen und alle Voxel in diesem 3D-Volumen anschauen.

`index_img(fmri_img, 0)` wählt das erste Volumen aus. Es wirkt unschärfer und weniger detailliert als das anatomische Bild, weil funktionelle Bilder schnell und immer wieder aufgenommen werden.

In [ ]:
first_volume = image.index_img(fmri_img, 0)

plotting.plot_epi(
    first_volume,
    title="Das erste 3D-Volumen im fMRT-Scan",
)
plotting.show()

### Was bedeuten die Farben?

In diesem Bild zeigen hellere und dunklere Farben die **rohe Signalintensität**, die in den Voxeln zu einem bestimmten Moment aufgenommen wurde.

Diese Farben zeigen **noch keine Aufgabenaktivierung**. Ein helles Voxel ist nicht automatisch ein Voxel, das bei einem Tastendruck oder einem Satz besonders beschäftigt war. Seine Intensität hängt auch von Gewebeart, Scannerempfindlichkeit, Bildposition und anderen Quellen von Variation ab.

Um etwas über eine Aufgabe zu sagen, müssen wir das sich verändernde Voxelsignal mit dem Experiment verbinden und **eine Bedingung mit einer anderen Bedingung vergleichen**.

## Das Localizer-Experiment

Um Aufgabenbedingungen zu vergleichen, verwenden wir Daten von einer Versuchsperson aus der Localizer-Studie von Pinel und Kolleginnen/Kollegen. Während eines kurzen Scans sah oder hörte die Person viele kurze Anweisungen und Reize.

<img src="https://raw.githubusercontent.com/raimund-buehler/visualizing-the-brain/main/src/images/pinel_localizer_tasks.png" alt="Beispiele für visuelle und auditive Aufgaben im Pinel-Localizer-Experiment" width="850">

Zu den Aufgaben gehörten:

- kurze Sätze lesen,
- kurze Sätze hören,
- einfache Subtraktionen sehen oder hören,
- nach einer visuellen oder gesprochenen Anweisung die linke oder rechte Taste drücken, und
- blinkende Schachbrettmuster anschauen.

Wenn du noch nie eine fMRT-Studie gesehen hast, wirken diese Bedingungen vielleicht etwas zufällig. Sie sind aber sorgfältig ausgewählt. Jede Bedingung hilft Forschenden, eine einfache Frage über eine wichtige Gehirnfunktion zu stellen.

Zum Beispiel helfen Schachbrettmuster bei Fragen zur **Vision**: Wie reagiert das Gehirn auf Dinge, die wir sehen? Tastendrücke helfen bei Fragen zur **Bewegung**: Wie sendet das Gehirn Befehle, um unsere Hände zu bewegen? Sätze helfen bei Fragen zur **Sprache**: Wie versteht das Gehirn gesprochene oder geschriebene Wörter? Subtraktionsaufgaben helfen bei Fragen zu **Zahlen**: Wie arbeitet das Gehirn mit einfacher Arithmetik?

Weil das Experiment mehrere Aufgabentypen enthält, können wir sie auf verschiedene Arten vergleichen. Zum Beispiel: Welche Areale reagieren stärker auf horizontale als auf vertikale Schachbrettmuster? Wie unterscheidet sich Hören von Lesen? Wie unterscheiden sich rechte und linke Tastendrücke?

Der Datensatz enthält außerdem eine **Ereignistabelle**, in der steht, was wann passiert ist. Mit dieser Tabelle können wir die Aufgabenbedingungen mit dem fMRT-Signal verbinden.

## Den Scan mit dem Experiment verbinden

Der Scanner nimmt nur sich verändernde Zahlen auf. Er weiß nicht, ob die Versuchsperson gerade gelesen, zugehört oder eine Taste gedrückt hat.

Die **Ereignistabelle** liefert diese fehlende Information. Jede Zeile beschreibt ein Ereignis:

- `trial_type`: was die Versuchsperson gemacht hat,
- `onset`: wann es begann, in Sekunden nach Scanbeginn,
- `duration`: wie lange es dauerte, in Sekunden.

Die Analyse verwendet diese Zeitpunkte, um das Experiment mit den 128 gemessenen Gehirnvolumen abzugleichen.

In [ ]:
# Zeige die ersten zehn Ereignisse im Experiment.
events.head(10)

## Das Modell anpassen

Jetzt wollen wir zwei Dinge verbinden:

1. das fMRT-Signal aus jedem Voxel, und
2. die Zeitpunkte der Aufgabenbedingungen.

Für jede Bedingung baut der Computer einen einfachen **Prädiktor**. Ein Prädiktor ist wie ein Hinweis, der sagt: "Wenn diese Aufgabe für dieses Voxel wichtig ist, sollte sich das Signal ungefähr zu diesen Zeiten verändern."

Eine hilfreiche Vorstellung sind Tonregler. Stell dir einen Regler für rechte Tastendrücke vor, einen für linke Tastendrücke, einen für Lesen, einen für Hören und so weiter. Für jedes Voxel fragt das Modell: Welche Regler müssen hochgedreht werden, damit wir das Signal dieses Voxels erklären können?

Das Modell gibt jedem Prädiktor eine Zahl, ein **Gewicht**. Die Mathematik brauchen wir hier nicht. Wichtig ist:

- ein größeres Gewicht bedeutet, dass diese Aufgabenzeitpunkte hilfreicher waren, um das Voxel zu erklären,
- ein Gewicht nahe null bedeutet, dass diese Aufgabenzeitpunkte kaum geholfen haben,
- das Modell wiederholt das für jedes Voxel im Gehirn.

Die Gehirnantwort ist langsam. Deshalb weiß Nilearn auch, dass das fMRT-Signal meist erst einige Sekunden nach einem Aufgabenereignis ansteigt, nicht sofort. Dieses Detail lassen wir Nilearn für uns erledigen.

Unser Ziel ist einfach: Wir wollen herausfinden, wo der **Prädiktor für die rechte Hand** wichtiger war als der **Prädiktor für die linke Hand**.

In [ ]:
# Beschreibe und passe das First-Level-Modell einer Versuchsperson an.
first_level_model = FirstLevelModel(
    t_r=2.4,
    slice_time_ref=0,
    hrf_model="spm",
    drift_model="cosine",
    high_pass=0.01,
    smoothing_fwhm=4,
    minimize_memory=False,
)

first_level_model = first_level_model.fit(fmri_img, events=events)
print("Modell erfolgreich angepasst.")

## Ein Kontrast: rechte Hand verglichen mit linker Hand

Nach dem Anpassen hat jedes Voxel ein Modellgewicht für jede Bedingung. Ein **Kontrast** vergleicht diese Gewichte.

Unsere Frage ist:

> Wo war die modellierte Antwort stärker für rechte Tastendrücke als für linke Tastendrücke?

Kurz gesagt:

```text
durchschnittliches Gewicht rechte Hand - durchschnittliches Gewicht linke Hand
```

Das Subtrahieren ist wichtig. Eine farbige Karte einer einzelnen Bedingung allein kann uns nicht sagen, ob die Antwort besonders stark für diese Bedingung ist. Ein Kontrast macht den Vergleich ausdrücklich.

In [ ]:
# Bilde für jede Hand den Durchschnitt über visuelle und auditive Anweisungen.
right_minus_left = (
    "0.5 * (visual_right_hand_button_press "
    "+ audio_right_hand_button_press) "
    "- 0.5 * (visual_left_hand_button_press "
    "+ audio_left_hand_button_press)"
)

right_minus_left_map = first_level_model.compute_contrast(
    right_minus_left,
    output_type="z_score",
)

print("Kontrast berechnet.")

### Die Kontrastkarte plotten

Der Plot unten visualisiert den Kontrast:

In [ ]:
# Finde das stärkste Voxel für rechte Hand > linke Hand und wandle es in Kartenkoordinaten um.
contrast_data = right_minus_left_map.get_fdata()
peak_index = np.unravel_index(np.nanargmax(contrast_data), contrast_data.shape)
peak_coord = right_minus_left_map.affine @ [*peak_index, 1]
peak_coord = peak_coord[:3]

plotting.plot_stat_map(
    right_minus_left_map,
    bg_img=anat_img,
    display_mode="ortho",
    cut_coords=peak_coord,
    threshold=3.0,
    symmetric_cbar=True,
    title="Fokusansicht: stärkster Cluster für rechte Hand > linke Hand",
)
plotting.show()

plotting.plot_stat_map(
    right_minus_left_map,
    bg_img=anat_img,
    display_mode="z",
    cut_coords=[40, 50, 60, 70],
    threshold=3.0,
    symmetric_cbar=True,
    title="Rechter Tastendruck minus linker Tastendruck",
)
plotting.show()

Schau dir die Seiten des Gehirns genau an:

1. Wo im Gehirn sehen wir die stärksten Antworten? Oben oder unten?
2. Wo sind die stärksten warmen Farben? Eher links oder rechts im Gehirn?
3. Wo sind die stärksten kühlen Farben? Eher links oder rechts im Gehirn?

### Was haben wir gerade entdeckt?

Die Kontrastkarte zeigt uns etwas Wichtiges darüber, wie das Gehirn den Körper steuert.

<img src="https://raw.githubusercontent.com/raimund-buehler/visualizing-the-brain/main/src/images/Anatomytool_Sensory_homunculus_English.jpg" alt="Motorischer Homunculus mit Körperteil-Repräsentation entlang des motorischen Kortex" width="650">

Dieses Bild nennt man einen **motorischen Homunculus**. Es ist eine vereinfachte Karte des motorischen Kortex. Verschiedene Teile dieses Streifens von Gehirngewebe sind mit Bewegungen verschiedener Körperteile verbunden. Die Hand hat einen großen Bereich, weil Hände viele genaue Bewegungen machen können.

In unserem Kontrast fragten wir:

```text
rechter Tastendruck - linker Tastendruck
```

Das bedeutet, dass wir die Farben so lesen sollten:

- **Warme Farben** bedeuten: mehr modellierte Aktivität für die **rechte Hand** als für die linke Hand.
- **Kühle Farben** bedeuten: weniger modellierte Aktivität für die **rechte Hand** als für die linke Hand. Anders gesagt: Diese Voxel reagierten stärker für die **linke Hand**.

Wir haben gerade ein wichtiges Prinzip der Gehirnorganisation entdeckt: **Kontralateralität**. Das bedeutet, dass eine Seite des Gehirns hauptsächlich mit der gegenüberliegenden Körperseite verbunden ist. Der linke motorische Kortex ist stark daran beteiligt, die rechte Hand zu bewegen. Der rechte motorische Kortex ist stark daran beteiligt, die linke Hand zu bewegen.

Solche Kreuzungen gibt es auch in anderen Gehirnsystemen. Zum Beispiel wird die linke Seite der visuellen Welt größtenteils von der rechten Gehirnseite verarbeitet, und die rechte Seite der visuellen Welt größtenteils von der linken Gehirnseite. Beim Hören ist es etwas gemischter, weil beide Ohren Informationen an beide Gehirnseiten senden. Die gegenüberliegende Seite ist aber oft besonders wichtig.

Das Ergebnis handelt also nicht nur von Tastendrücken. Es ist ein Beispiel für eine größere Idee: Das Gehirn ist in Karten organisiert, und viele dieser Karten kreuzen von einer Seite des Körpers oder der Welt zur anderen Seite des Gehirns.

## Quellen und weiterführende Links

Dieses Notebook passt Lehrmaterial und Abbildungen aus `TEWA2/02_Understanding_MRI_Data.ipynb` und `TEWA2/06_First_Level_Analysis.ipynb` in diesem Repository an.

- [Nilearn: 3D- und 4D-Bilder](https://nilearn.github.io/stable/auto_examples/00_tutorials/plot_3d_and_4d_niimg.html)
- [Nilearn: First-Level-Modelle](https://nilearn.github.io/stable/glm/first_level_model.html)
- [Nilearn-Plotting-Werkzeuge](https://nilearn.github.io/stable/modules/plotting.html)
- Pinel, P., Thirion, B., Meriaux, S. et al. (2007). Fast reproducible identification and large-scale databasing of individual functional cognitive networks. *BMC Neuroscience, 8*, 91. https://doi.org/10.1186/1471-2202-8-91